# HW3 Image Classification
## We strongly recommend that you run with Kaggle for this homework


# Get Data
Notes: if the links are dead, you can download the data directly from Kaggle and upload it to the workspace, or you can use the Kaggle API to directly download the data into colab.


In [ ]:
!wget https://www.dropbox.com/s/6l2vcvxl54b0b6w/food11.zip

In [ ]:
!unzip food11.zip

# Training

In [ ]:
# Import necessary packages.
import numpy as np
import pandas as pd
import torch
import os
import torch.nn as nn
import torchvision.transforms.v2 as transforms
from PIL import Image
# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Subset, Dataset
from torchvision.datasets import DatasetFolder, VisionDataset

# This is for the progress bar.
from tqdm.auto import tqdm
import random

In [ ]:
myseed = 6666  # set a random seed for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(myseed)
random.seed(myseed)
torch.manual_seed(myseed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(myseed)

## **Transforms**
Torchvision provides lots of useful utilities for image preprocessing, data wrapping as well as data augmentation.

Please refer to PyTorch official website for details about different transforms.

In [ ]:
# Normally, We don't need augmentations in testing and validation.
# All we need here is to resize the PIL image and transform it into Tensor.
test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToImage(), 
    transforms.ToDtype(torch.float32, scale=True),
    # Image Normalization
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# However, it is also possible to use augmentation in the testing phase.
# You may use train_tfm to produce a variety of images and then test using ensemble methods
train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(128, antialias=True),
    transforms.RandomHorizontalFlip(0.5),
    transforms.TrivialAugmentWide(),
    transforms.ToImage(), 
    transforms.ToDtype(torch.float32, scale=True),
    # Image Normalization
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


CutMix & MixUp Transform

In [ ]:
from torch.utils.data import default_collate
import torch.nn.functional as F

cutmix = transforms.CutMix(num_classes=11)
mixup = transforms.MixUp(num_classes=11)
cutmix_or_mixup = transforms.RandomChoice([cutmix, mixup])

def collate_fn(batch):
    return cutmix_or_mixup(*default_collate(batch))

## **Datasets**
The data is labelled by the name, so we load images and label while calling '__getitem__'

In [ ]:
from pathlib import Path
from typing import Literal
class FoodDataset(Dataset):
    def __init__(self,files: list[Path], tfm=test_tfm, split: Literal["train", "val", "test"]="train"):
        super(FoodDataset).__init__()
        self.files = files
        print(f"One sample: ",self.files[0])
        self.transform = tfm
        self.split = split
        self.data = dict()
        for f in self.files:
            self.data[str(f)] = self.transform(Image.open(f))
  
    def __len__(self):
        return len(self.files)
  
    def __getitem__(self,idx: int):
        fname = self.files[idx]
        if self.split in ["train", "val"]:
            label = int(fname.name.split("_")[0])
            return self.data[str(fname)],label
        else:
            return self.data[str(fname)]


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        # 投影分支：当stride!=1或通道数变化时对齐维度
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(identity)
        out = self.relu(out + identity)
        return out

class ResidualNet(nn.Module):
    def __init__(self, num_classes=11):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        # 128x128
        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64, stride=1),
            ResidualBlock(64, 64, stride=1),
        )  # -> 128x128
        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),   # -> 64x64
            ResidualBlock(128, 128, stride=1),
        )
        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),  # -> 32x32
            ResidualBlock(256, 256, stride=1),
        )
        self.head = nn.Sequential(
            nn.AdaptiveMaxPool2d((4, 4)),       # 与你原fc尺寸一致
            nn.Flatten(),
            nn.Linear(256*4*4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.head(x)
        return x

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        # torch.nn.MaxPool2d(kernel_size, stride, padding)
        # input 維度 [3, 128, 128]
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),  # [64, 128, 128]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [64, 64, 64]

            nn.Conv2d(64, 128, 3, 1, 1), # [128, 64, 64]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [128, 32, 32]

            nn.Conv2d(128, 256, 3, 1, 1), # [256, 32, 32]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [256, 16, 16]

            nn.Conv2d(256, 512, 3, 1, 1), # [512, 16, 16]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 8, 8]
            
            nn.Conv2d(512, 512, 3, 1, 1), # [512, 8, 8]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 4, 4]
        )
        self.fc = nn.Sequential(
            nn.Linear(512*4*4, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 11)
        )

    def forward(self, x):
        out = self.cnn(x)
        out = out.view(out.size()[0], -1)
        return self.fc(out)

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
class ResNet50(nn.Module):
    def __init__(self, num_classes=11):
        super(ResNet50, self).__init__()
        
        # 加载 ResNet50 模型（去掉最后的 fc 层）
        resnet = resnet50()  # 使用 weights=ResNet50_Weights.IMAGENET1K_V1 加载预训练权重
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])  # 去掉 fc 层
        
        # 获取原始 fc 层的输入特征数
        num_features = resnet.fc.in_features
        
        # 定义新的 fc 层
        self.fc = nn.Sequential(
            nn.Linear(num_features, 1024),
            nn.BatchNorm1d(1024),  # Batch Normalization
            nn.ReLU(),
            nn.Dropout(0.5),  # Dropout

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),  # Batch Normalization
            nn.ReLU(),
            nn.Dropout(0.5),  # Dropout

            nn.Linear(512, num_classes)  # 输出层
        )

    def forward(self, x):
        out = self.cnn(x)
        out = torch.flatten(out, 1)  # 使用 torch.flatten 代替 view
        out = self.fc(out)
        return out

In [ ]:
batch_size = 64
_dataset_dir = "./food11"

# N-fold cross validation
def split_dataset(files, valid_fold, n_splits=4):
    assert valid_fold < n_splits, "Fold index must be less than number of splits."
    random.shuffle(files)
    fold_size = len(files) // n_splits
    valid_files = files[valid_fold * fold_size:(valid_fold + 1) * fold_size]
    train_files = [f for f in files if f not in valid_files]
    return train_files, valid_files

data_files = (list(Path(_dataset_dir).joinpath("training").glob("*.jpg"))
              + list(Path(_dataset_dir).joinpath("validation").glob("*.jpg")))
valid_fold = 0  # You can change this to 0,1,2,3 for 4-fold cross validation
n_splits = 4
train_files, valid_files = split_dataset(data_files, valid_fold, n_splits)
train_set = FoodDataset(train_files, train_tfm)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True, collate_fn=collate_fn)
valid_set = FoodDataset(valid_files, test_tfm)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)


_exp_name = f"mixup_{valid_fold}"


In [ ]:
imgs, labels = next(iter(train_loader))
print("imgs.shape:", imgs.shape)  # 应该是 (B, C, H, W)
print("labels.shape:", labels.shape)  # mixup是 (B, 11) 否则是 (B,)
print("labels[0]:", labels[0])  # 打印第一个样本的标签，看看是 one-hot 还是整数标签

In [ ]:
# "cuda" only when GPUs are available.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print(f"Fold {valid_fold} out of {n_splits} folds")

# The number of training epochs and patience.
n_epochs = 500
patience = 500 # If no improvement in 'patience' epochs, early stop

# Initialize a model, and put it on the device specified.
# model = ResNet50().to(device)
model = Classifier().to(device)
# 如果接续上次训练 取消下面这行代码的注释
# model.load_state_dict(torch.load(f"{_exp_name}_last.ckpt", map_location=device))

# For the classification task, we use cross-entropy as the measurement of performance.
criterion = nn.CrossEntropyLoss(label_smoothing=0.08)

# Initialize optimizer, you may fine-tune some hyperparameters such as learning rate on your own.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) 

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=0.0001)
# Initialize trackers, these are not parameters and should not be changed
stale = 0
best_acc = 0

for epoch in range(n_epochs):

    # ---------- Training ----------
    # Make sure the model is in train mode before training.
    model.train()

    # These are used to record information in training.
    train_loss = []
    train_accs = []

    for batch in tqdm(train_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch
        #imgs = imgs.half()
        #print(imgs.shape,labels.shape)

        # Forward the data. (Make sure data and model are on the same device.)
        logits = model(imgs.to(device))

        # Calculate the cross-entropy loss.
        # We don't need to apply softmax before computing cross-entropy as it is done automatically.
        # --------- 根据 label 形状选择 loss & acc ---------
        if labels.dim() == 2 and labels.size(1) != 1:
            # Mixup/CutMix: labels 为 (B, C) soft label
            targets = labels.to(device).float()           # (B, 11)
            loss_prob = F.log_softmax(logits, dim=1)
            loss = -(targets * loss_prob).sum(dim=1).mean()
            
            # 软准确率（可以保留）
            probs = torch.softmax(logits, dim=1)
            acc = (probs * targets).sum(dim=1).mean().item()
        else:
            # 普通情况: labels 为 (B,) long
            loss = criterion(logits, labels.to(device))
            acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean().item()
        # -------------------------------------------------

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)
        optimizer.step()
        scheduler.step()

        train_loss.append(loss.item())
        train_accs.append(acc)
        
    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)

    # Print the information.
    print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

    # ---------- Validation ----------
    # Make sure the model is in eval mode so that some modules like dropout are disabled and work normally.
    model.eval()

    # These are used to record information in validation.
    valid_loss = []
    valid_accs = []

    # Iterate the validation set by batches.
    for batch in tqdm(valid_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch
        #imgs = imgs.half()

        # We don't need gradient in validation.
        # Using torch.no_grad() accelerates the forward process.
        with torch.no_grad():
            logits = model(imgs.to(device))

        # We can still compute the loss (but not the gradient).
        loss = criterion(logits, labels.to(device))

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # Record the loss and accuracy.
        valid_loss.append(loss.item())
        valid_accs.append(acc)
        #break

    # The average loss and accuracy for entire validation set is the average of the recorded values.
    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)

    # Print the information.
    print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")


    # update logs
    if valid_acc > best_acc:
        with open(f"./{_exp_name}_log.txt","a"):
            print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f} -> best")
    else:
        with open(f"./{_exp_name}_log.txt","a"):
            print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")


    # save models
    if valid_acc > best_acc:
        print(f"Best model found at epoch {epoch}, saving model")
        torch.save(model.state_dict(), f"{_exp_name}_best.ckpt") # only save best to prevent output memory exceed error
        best_acc = valid_acc
        stale = 0
    else:
        stale += 1
        if stale > patience:
            print(f"No improvment {patience} consecutive epochs, early stopping")
            break

如果效果还不理想，需要增加训练时间，运行以下代码将模型保存下来。

In [ ]:
torch.save(model.state_dict(), f"{_exp_name}_last.ckpt")

# Testing and generate prediction CSV

使用训练好的模型直接进行测试，并生成提交文件。

In [ ]:
test_files = sorted(list(Path(_dataset_dir).joinpath("test").glob("*.jpg")))
test_set = FoodDataset(test_files, tfm=test_tfm, split="test")
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

# model_best = ResNet50().to(device)
model_best = Classifier().to(device)
model_best.load_state_dict(torch.load(f"{_exp_name}_best.ckpt", map_location=device))
model_best.eval()

prediction = []
with torch.no_grad():
    for data in test_loader:
        logits: torch.Tensor = model_best(data.to(device))      # (B, 11)
        preds = logits.argmax(dim=1).cpu().numpy()              # (B,)
        prediction.extend(preds.tolist())

print("num test images:", len(test_set))
print("num predictions:", len(prediction))
assert len(prediction) == len(test_set)

def pad4(i: int) -> str:
    return "0"*(4-len(str(i)))+str(i)

df = pd.DataFrame()
df["Id"] = [pad4(i) for i in range(1, len(test_set)+1)]
df["Category"] = prediction
df.to_csv(f"{_exp_name}_submission.csv", index=False)
print(f"saved {_exp_name}_submission.csv")

Test Time Augmentation (TTA)

Test Time Augmentation (TTA) is a technique used to improve the performance of a model during inference by applying various transformations to the input data and averaging the predictions. This can help to reduce overfitting and improve the generalization of the model.

In [ ]:
test_loaders = []
test_loaders.append(test_loader)
for _ in range(5):
    test_set_i = FoodDataset(test_files, tfm=train_tfm, split="test")
    test_loaders.append(DataLoader(test_set_i, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True))
    

In [ ]:
# model_best = Classifier().to(device)
model_best = ResNet50().to(device)
model_best.load_state_dict(torch.load(f"{_exp_name}_best.ckpt"))
model_best.eval()
prediction = []
with torch.no_grad():
    for data in test_loader:
        test_pred = model_best(data.to(device))
        test_label = np.argmax(test_pred.cpu().data.numpy(), axis=1)
        prediction += test_label.squeeze().tolist()

In [ ]:
preds = [list() for _ in range(6)]
for i, test_loader in enumerate(test_loaders):
    model_best.eval()
    with torch.no_grad():
        for j, data in enumerate(test_loader):
            test_pred = model_best(data.to(device))
            preds[i].extend(test_pred.cpu())
preds[-1] = prediction
preds_np = np.stack(preds, axis=0)  # shape: (6, 3347, 11)
# 对6个预测结果加权平均
weights = [0.1, 0.1, 0.1, 0.1, 0.1, 0.5]  # 可以根据需要调整权重
final_preds = np.tensordot(weights, preds_np, axes=([0], [0]))  # shape: (3347, 11)
prediction = np.argmax(final_preds, axis=1)     # shape: (3347,)
np.save(f"{_exp_name}_final_preds.npy", final_preds)


集成预测结果

In [ ]:
if all(Path(f"mixup_{i}_final_preds.npy").exists() for i in range(n_splits)):
    final_preds_list = [np.load(f"mixup_{i}_final_preds.npy") for i in range(n_splits)]
    ensemble_final_preds = np.mean(final_preds_list, axis=0)  # 对4个fold的预测结果取平均 shape: (3347, 11)
    prediction = np.argmax(ensemble_final_preds, axis=1)        # shape: (3347,)
    #create test csv
    def pad4(i):
        return "0"*(4-len(str(i)))+str(i)
    df = pd.DataFrame()
    df["Id"] = [pad4(i) for i in range(1,len(test_set)+1)]
    df["Category"] = prediction
    df.to_csv("submission.csv",index = False)

# Q1. Augmentation Implementation
## Implement augmentation by finishing train_tfm in the code with image size of your choice. 
## Directly copy the following block and paste it on GradeScope after you finish the code
### Your train_tfm must be capable of producing 5+ different results when given an identical image multiple times.
### Your  train_tfm in the report can be different from train_tfm in your training code.


In [ ]:
train_tfm = transforms.Compose([
    # Resize the image into a fixed shape (height = width = 128)
    transforms.Resize((128, 128)),
    # You need to add some transforms here.
    transforms.RandomHorizontalFlip(0.5),
    transforms.TrivialAugmentWide(),
    transforms.ToImage(), 
    transforms.ToDtype(torch.float32, scale=True),
])

展示变换效果

In [ ]:
from PIL import Image
test_files = list(Path(_dataset_dir).joinpath("test").glob("*.jpg"))

img = Image.open(random.choice(test_files))
display(img)
for _ in range(5):
    img_tfm = train_tfm(img)
    img_tfm_pil = transforms.ToPILImage()(img_tfm)
    display(img_tfm_pil)

# Q2. Residual Implementation
![](https://i.imgur.com/GYsq1Ap.png)
## Directly copy the following block and paste it on GradeScope after you finish the code


In [ ]:
from torch import nn
class Residual_Network(nn.Module):
    def __init__(self):
        super(Residual_Network, self).__init__()
        
        self.cnn_layer1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
        )

        self.cnn_layer2 = nn.Sequential(
            nn.Conv2d(64, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
        )

        self.cnn_layer3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
        )

        self.cnn_layer4 = nn.Sequential(
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
        )
        self.cnn_layer5 = nn.Sequential(
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.BatchNorm2d(256),
        )
        self.cnn_layer6 = nn.Sequential(
            nn.Conv2d(256, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
        )
        self.fc_layer = nn.Sequential(
            nn.Linear(256* 32* 32, 256),
            nn.ReLU(),
            nn.Linear(256, 11)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        # input (x): [batch_size, 3, 128, 128]
        # output: [batch_size, 11]

        # Extract features by convolutional layers.
        x1 = self.cnn_layer1(x)
        
        x1 = self.relu(x1)          # x1: [B, 64, 128, 128]
        
        x2 = self.cnn_layer2(x1)
        
        x2 = self.relu(x1 + x2)     # x2: [B, 64, 128, 128], 残差连接
        
        x3 = self.cnn_layer3(x2)
        
        x3 = self.relu(x3)          # x3: [B, 128, 64, 64]
        
        x4 = self.cnn_layer4(x3)
        
        x4 = self.relu(x3 + x4)     # x4: [B, 128, 64, 64], 残差连接
        
        x5 = self.cnn_layer5(x4)
        
        x5 = self.relu(x5)          # x5: [B, 256, 32, 32]
        
        x6 = self.cnn_layer6(x5)
        
        x6 = self.relu(x5 + x6)     # x6: [B, 256, 32, 32], 残差连接
        
        # The extracted feature map must be flatten before going to fully-connected layers.
        xout = x6.flatten(1)

        # The features are transformed by fully-connected layers to obtain the final logits.
        xout = self.fc_layer(xout)
        return xout